In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
import statsmodels.api as sm
import numpy as np




In [ ]:
#Subimos la base de datos
df = pd.read_csv('/content/properati-MX-2017-07-01-properties-sell.csv', encoding='ISO-8859-1', engine='python', on_bad_lines='skip')
df.info()

FileNotFoundError: [Errno 2] No such file or directory: '/content/properati-MX-2017-07-01-properties-sell.csv'

In [ ]:
# Usamos .str.contains para que busque "Puebla" sin importar lo que haya antes o después
df_puebla = df[df['place_with_parent_names'].str.contains('Puebla', na=False)].copy()
len(df_puebla)

In [ ]:
#Renombramos columnas
df_puebla = df_puebla.rename(columns={
    'surface_total_in_m2': 'superficie_total',
    'surface_covered_in_m2': 'superficie_cubierta',
    'floor': 'pisos',
    'rooms': 'cuartos',
    'price_aprox_local_currency': 'precio_mxn'
})


In [ ]:
#Eliminamos las columnas que no usamos
v_interes = ['precio_mxn', 'superficie_total', 'superficie_cubierta', 'pisos', 'cuartos','lat', 'lon']
df_puebla = df_puebla[v_interes].copy()
df_puebla

In [ ]:
# Eliminamos filas sin precio (nuestra Y),sin superficie, y sin lat y lon,(nuestra X principal)
df_puebla = df_puebla.dropna(subset=['precio_mxn', 'superficie_total', 'superficie_cubierta','lat','lon'	])
len(df_puebla)

In [ ]:
# Para baños y recámaras, podemos rellenar con 1s
df_puebla['cuartos'] = df_puebla['cuartos'].fillna(1)
df_puebla['pisos'] = df_puebla['pisos'].fillna(1)

In [ ]:
#Filtro de Outliers (Valores atípicos)
# Quitamos "ruido": Precios menores a 300k o mayores a 20M, y superficies irreales
df_puebla = df_puebla[(df_puebla['precio_mxn'] > 300000) & (df_puebla['precio_mxn'] < 20000000)]
df_puebla = df_puebla[(df_puebla['superficie_total'] > 35) & (df_puebla['superficie_total'] < 1000)]
df_puebla = df_puebla[(df_puebla['superficie_cubierta'] > 35) & (df_puebla['superficie_cubierta'] < 1000)]

In [ ]:
df_puebla = df_puebla[(df_puebla['superficie_total'] > 0) & (df_puebla['precio_mxn'] > 0)]
df_puebla

In [ ]:
df_final=df_puebla

In [ ]:
print(f"Limpieza terminada. Registros finales de Puebla: {len(df_final)}")

In [ ]:
# Configuramos el estilo y el tamaño de la figura
plt.style.use('seaborn-v0_8-muted')
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Distribución de Variables - Vivienda en Puebla', fontsize=16)

# 1. Histograma de Precios (Y)
sns.histplot(df_final['precio_mxn'], bins=30, kde=True, ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('Distribución de Precios')

# 2. Histograma de Superficie (X1)
sns.histplot(df_final['superficie_total'], bins=30, kde=True, ax=axes[0, 1], color='salmon')
axes[0, 1].set_title('Distribución de Superficie Total (m2)')

# 3. Histograma de Cuartos (X2)
sns.histplot(df_final['cuartos'], bins=10, ax=axes[1, 0], color='green')
axes[1, 0].set_title('Distribución de Cuartos (Habitaciones)')

# 4. Histograma de Pisos (X3)
sns.histplot(df_final['pisos'], bins=5, ax=axes[1, 1], color='purple')
axes[1, 1].set_title('Distribución de Pisos (Niveles)')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
# ESTADÍSTICOS, CORRELACIONES Y BOXPLOTS

# Revisión de valores faltantes final
print(" Valores Faltantes ")
print(df_final.isnull().sum())
print("\n")

# Estadísticos descriptivos
print("Estadísticos Descriptivos")
display(df_final.describe())


In [ ]:

# Matriz de correlaciones
plt.figure(figsize=(8, 6))
# Dejamos fuera lat y lon para enfocarnos en las características del inmueble
v_corr = ['precio_mxn', 'superficie_total', 'superficie_cubierta', 'cuartos', 'pisos']
matriz_corr = df_final[v_corr].corr()

sns.heatmap(matriz_corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Matriz de Correlaciones - Variables Estructurales')
plt.show()



In [ ]:
# Distribuciones mediante Boxplots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
sns.boxplot(y=df_final['precio_mxn'], ax=axes[0], color='skyblue').set_title('Boxplot: Precio (MXN)')
sns.boxplot(y=df_final['superficie_total'], ax=axes[1], color='salmon').set_title('Boxplot: Superficie Total')
sns.boxplot(y=df_final['cuartos'], ax=axes[2], color='green').set_title('Boxplot: Cuartos')
plt.tight_layout()
plt.show()

In [ ]:
# MODELO BASE

# Definimos la variable dependiente (Y)
Y = df_final['precio_mxn']

# Definimos las variables independientes (X)
X = df_final[['superficie_total', 'pisos', 'lat', 'lon']]

# Agregar el intercepto al modelo
X = sm.add_constant(X)

modelo_base_1 = sm.OLS(Y, X).fit()

print(modelo_base_1.summary())



In [ ]:
#Diagnosticos

#Breusch-Pagan (Heteroscedasticidad)
# H0: No hay heteroscedasticidad
# H1: hay heteroscedasticidad
bp_test = sms.het_breuschpagan(modelo_base_1.resid, modelo_base_1.model.exog)
print("\n2. Prueba de Breusch-Pagan para Heteroscedasticidad:")
print(f"Estadístico LM: {bp_test[0]:.4f}")
print(f"Valor p (LM): {bp_test[1]:.4e}")
if bp_test[1] < 0.05:
    print("Conclusión: Se rechaza H0 (Hay evidencia de Heteroscedasticidad).")
else:
    print("Conclusión: No se rechaza H0 (Hay Homoscedasticidad).")
print("-" * 50)



#Haciendo cambios de varibales y nuevos ajustes con las distancias, y superficies

In [ ]:
#Limpiamos la base de datos
df_final = df_final.dropna(subset=['precio_mxn', 'superficie_total', 'lat', 'lon'])

#Tomamos en cuenta Puebla Capital, para el calculo de la nueva variable de distancial al centro
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0

    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))

    return R * c

zocalo_lat, zocalo_lon = 19.0436, -98.1981

df_final['distancia_centro_km'] = df_final.apply(
    lambda row: haversine(row['lat'], row['lon'], zocalo_lat, zocalo_lon),
    axis=1
)



#Mapa
lat_center = df_final['lat'].mean()
lon_center = df_final['lon'].mean()

fig = px.scatter_mapbox(
    df_final,
    lat="lat",
    lon="lon",
    color="precio_mxn",
    size="superficie_total",
    center={"lat": lat_center, "lon": lon_center},
    zoom=10,
    hover_data=["precio_mxn", "cuartos", "distancia_centro_km"],
    title="Geolocalización de Viviendas en Puebla"
)

fig.update_layout(mapbox_style="open-street-map")
fig.show()


In [ ]:
#Ajuste Precio-Distancia
plt.scatter(df_final['distancia_centro_km'], df_final['precio_mxn'])
plt.xlabel("Distancia al centro (km)")
plt.ylabel("Precio (MXN)")
plt.title("Relación Precio vs Distancia al Zócalo")
plt.show()

In [ ]:

#Modelo
if 'cuartos' not in df_final.columns:
    df_final['cuartos'] = 1

y = df_final['precio_mxn']

X = df_final[['cuartos', 'superficie_total', 'lat', 'lon', 'distancia_centro_km']]

X = X.dropna()
y = y.loc[X.index]

X = sm.add_constant(X)

modelo_base_2 = sm.OLS(y, X).fit()

print(modelo_base_2.summary())

In [ ]:
#Diagnosticos
#Breusch-Pagan (Heteroscedasticidad)
# H0: No hay heteroscedasticidad
# H1: hay heteroscedasticidad
bp_test = sms.het_breuschpagan(modelo_base_2.resid, modelo_base_2.model.exog)
print("\n2. Prueba de Breusch-Pagan para Heteroscedasticidad:")
print(f"Estadístico LM: {bp_test[0]:.4f}")
print(f"Valor p (LM): {bp_test[1]:.4e}")
if bp_test[1] < 0.05:
    print("Conclusión: Se rechaza H0 (Hay evidencia de Heteroscedasticidad).")
else:
    print("Conclusión: No se rechaza H0 (Hay Homoscedasticidad).")
print("-" * 50)

#Segunda opción con outliers

In [ ]:
#Outliers
Q1 = df_final['precio_mxn'].quantile(0.25)
Q3 = df_final['precio_mxn'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df_final = df_final[(df_final['precio_mxn'] >= lower_bound) & (df_final['precio_mxn'] <= upper_bound)]

#Correlaciones
corr_matrix = df_final[['precio_mxn', 'superficie_total', 'distancia_centro_km']].corr()
print(corr_matrix)
df_final


In [ ]:
#Modelo 3
if 'cuartos' not in df_final.columns:
    df_final['cuartos'] = 0

y = df_final['precio_mxn']

X = df_final[['cuartos', 'superficie_total', 'lat', 'lon', 'distancia_centro_km']]

X = X.dropna()
y = y.loc[X.index]

X = sm.add_constant(X)

modelo_base_3 = sm.OLS(y, X).fit()

print(modelo_base_3.summary())


In [ ]:
#Diagnosticos
#Breusch-Pagan (Heteroscedasticidad)
# H0: No hay heteroscedasticidad
# H1: hay heteroscedasticidad
bp_test = sms.het_breuschpagan(modelo_base_3.resid, modelo_base_3.model.exog)
print("\n2. Prueba de Breusch-Pagan para Heteroscedasticidad:")
print(f"Estadístico LM: {bp_test[0]:.4f}")
print(f"Valor p (LM): {bp_test[1]:.4e}")
if bp_test[1] < 0.05:
    print("Conclusión: Se rechaza H0 (Hay evidencia de Heteroscedasticidad).")
else:
    print("Conclusión: No se rechaza H0 (Hay Homoscedasticidad).")
print("-" * 50)

In [ ]:
#Modelo sin lat y lon

if 'cuartos' not in df_final.columns:
    df_final['cuartos'] = 0

y = df_final['precio_mxn']

X = df_final[['cuartos', 'superficie_total', 'distancia_centro_km']]

X = X.dropna()
y = y.loc[X.index]

X = sm.add_constant(X)

modelo_base_4 = sm.OLS(y, X).fit()

print(modelo_base_4.summary())

In [ ]:
#Diagnosticos
#Breusch-Pagan (Heteroscedasticidad)
# H0: No hay heteroscedasticidad
# H1: hay heteroscedasticidad
bp_test = sms.het_breuschpagan(modelo_base_4.resid, modelo_base_4.model.exog)
print("\n2. Prueba de Breusch-Pagan para Heteroscedasticidad:")
print(f"Estadístico LM: {bp_test[0]:.4f}")
print(f"Valor p (LM): {bp_test[1]:.4e}")
if bp_test[1] < 0.05:
    print("Conclusión: Se rechaza H0 (Hay evidencia de Heteroscedasticidad).")
else:
    print("Conclusión: No se rechaza H0 (Hay Homoscedasticidad).")
print("-" * 50)

In [ ]:
#MODELO LOG

# Creamos un nuevo DataFrame para las variables transformadas
df_log = df_final.copy()

# Aplicamos la transformación logarítmica
df_log['precio_mxn_log'] = np.log1p(df_log['precio_mxn'])
df_log['superficie_total_log'] = np.log1p(df_log['superficie_total'])

# Definimos la variable dependiente (Y) log-transformada
y_log = df_log['precio_mxn_log']

# Definimos las variables independientes (X) log-transformadas
X_log = df_log[['cuartos', 'superficie_total_log', 'distancia_centro_km']]

# Eliminamos cualquier fila con valores nulos que puedan haber surgido (aunque ya se limpiaron antes)
X_log = X_log.dropna()
y_log = y_log.loc[X_log.index]

# Agregamos el intercepto al modelo
X_log = sm.add_constant(X_log)

# Ajustamos el nuevo modelo OLS
modelo_log = sm.OLS(y_log, X_log).fit()

# Imprimimos el resumen del modelo
print(modelo_log.summary())

Parece que hemos encontrado un buen modelo, haremos todas las pruebas:

In [ ]:
print("\n--- Diagnósticos para modelo_log ---")

# 1. VIF
exog_data = modelo_log.model.exog
vif_data = pd.DataFrame()
vif_data["variable"] = modelo_log.model.exog_names
vif_data["VIF"] = [variance_inflation_factor(exog_data, i) for i in range(exog_data.shape[1])]

print("\n1. VIF para Multicolinealidad:")
display(vif_data)
print("\n Notas: Valores de VIF superiores a 5 o 10 suelen indicar problemas de multicolinealidad.")
for index, row in vif_data.iterrows():
    if row['variable'] == 'const':
        print(f"  Conclusión para {row['variable']}: El VIF para la constante es alto ({row['VIF']:.2f}), pero esto es esperado y no indica multicolinealidad entre las variables explicativas.")
    elif row['VIF'] >= 5:
        print(f"  Conclusión para {row['variable']}: Se sugiere posible multicolinealidad (VIF = {row['VIF']:.2f}).")
    else:
        print(f"  Conclusión para {row['variable']}: No hay evidencia de multicolinealidad significativa (VIF = {row['VIF']:.2f}).")
print("-" * 50)

# 2. Breusch-Pagan (Heteroscedasticidad)
# H0: No hay heteroscedasticidad
# H1: hay heteroscedasticidad
bp_test = sms.het_breuschpagan(modelo_log.resid, modelo_log.model.exog)
print("\n2. Prueba de Breusch-Pagan para Heteroscedasticidad:")
print(f"Estadístico LM: {bp_test[0]:.4f}")
print(f"Valor p (LM): {bp_test[1]:.4e}")
if bp_test[1] < 0.05:
    print("Conclusión: Se rechaza H0 (Hay evidencia de Heteroscedasticidad).")
else:
    print("Conclusión: No se rechaza H0 (Hay Homoscedasticidad).")
print("-" * 50)

# 3. Ramsey RESET (Especificación)
# H0: El modelo está correctamente especificado (no hay variables omitidas / relación lineal es adecuada)
# H1: El modelo tiene errores de especificación
try:
    # Correcting 'degree' to 'power'
    reset_test = linear_reset(modelo_log, power=3)
    print("\n3. Prueba Ramsey RESET:")
    print(reset_test)
    if reset_test.pvalue < 0.05:
        print("Conclusión: Se rechaza H0 (Posible error de especificación o no linealidad).")
    else:
        print("Conclusión: No se rechaza H0 (Modelo correctamente especificado).")
except Exception as e:
    print("\n3. Prueba Ramsey RESET:")
    print(f"No se pudo ejecutar el test RESET. Error: {e}")
print("-" * 50)

##CONCLUSIONES
VIF (Factor de Inflación de la Varianza) para Multicolinealidad:

Los valores VIF para cuartos (1.136), superficie_total_log (1.003) y distancia_centro_km (1.133) son muy bajos (cercanos a 1). Esto indica que no hay problemas significativos de multicolinealidad entre tus variables predictoras. El valor alto para const es normal y no es una preocupación.
Prueba de Breusch-Pagan para Heteroscedasticidad:

El estadístico LM es 5.5264 y el valor p es 0.13707. Dado que el valor p (0.13707) es mayor que 0.05, no se rechaza la hipótesis nula. Esto significa que no hay evidencia de heteroscedasticidad en el modelo, lo cual es un buen resultado y sugiere que los errores tienen varianza constante.
Prueba Ramsey RESET (Errores de Especificación):

El estadístico de prueba es 217.755 y el valor p es extremadamente bajo (5.189e-48). Dado que el valor p es menor que 0.05, se rechaza la hipótesis nula. Esto sugiere que el modelo podría tener un error de especificación o que la relación lineal no es la más adecuada. Podría ser necesario incluir más variables relevantes o considerar una forma funcional diferente para las variables existentes (por ejemplo, términos cuadráticos o interacciones).